In [1]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Create tables
cursor.execute("""
CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,
    name TEXT,
    email TEXT,
    city TEXT
)
""")

cursor.execute("""
CREATE TABLE enrollments (
    enrollment_id INTEGER PRIMARY KEY,
    student_id INTEGER,
    course_name TEXT,
    enrollment_date TEXT
)
""")

cursor.execute("""
CREATE TABLE grades (
    grade_id INTEGER PRIMARY KEY,
    student_id INTEGER,
    course_name TEXT,
    score INTEGER
)
""")

In [2]:
students_data = [
    (1, 'Alice', 'alice@email.com', 'Mumbai'),
    (2, 'Bob', 'bob@email.com', 'Delhi'),
    (3, 'Carol', 'carol@email.com', 'Bangalore'),
    (4, 'David', 'david@email.com', 'Chennai'),
    (5, 'Emma', 'emma@email.com', 'Pune')
]

cursor.executemany("INSERT INTO students VALUES (?, ?, ?, ?)", students_data)


enrollments_data = [
    (101, 1, 'Python', '2024-01-15'),
    (102, 1, 'SQL', '2024-02-01'),
    (103, 2, 'Data Science', '2024-01-20'),
    (104, 3, 'Python', '2024-02-10'),
    (105, 5, 'Machine Learning', '2024-01-25')
]

cursor.executemany("INSERT INTO enrollments VALUES (?, ?, ?, ?)", enrollments_data)


grades_data = [
    (201, 1, 'Python', 85),
    (202, 1, 'SQL', 90),
    (203, 2, 'Data Science', 78),
    (204, 4, 'Web Development', 88)
]

cursor.executemany("INSERT INTO grades VALUES (?, ?, ?, ?)", grades_data)

# Commit changes
conn.commit()

TASK1: Basic Joins (INNER and LEFT)

In [3]:
cursor.execute("""
SELECT s.name, e.course_name
FROM students AS s
INNER JOIN enrollments AS e
ON s.student_id = e.student_id;
""")

results = cursor.fetchall()
for row in results:
    print(row)

('Alice', 'Python')
('Alice', 'SQL')
('Bob', 'Data Science')
('Carol', 'Python')
('Emma', 'Machine Learning')


Explanation

This query returns student names and the courses they are enrolled in.

An INNER JOIN only returns rows where a match exists in both tables.

So:

If a student has an enrollment record → they appear in the result.

If a student has no enrollment → they do not appear.

| name  | course_name      |
| ----- | ---------------- |
| Alice | Python           |
| Alice | SQL              |
| Bob   | Data Science     |
| Carol | Python           |
| Emma  | Machine Learning |


In [4]:
cursor.execute("""SELECT s.name, e.course_name
FROM students AS s
LEFT JOIN enrollments AS e
ON s.student_id = e.student_id;""")

results = cursor.fetchall()
for row in results:
    print(row)

('Alice', 'Python')
('Alice', 'SQL')
('Bob', 'Data Science')
('Carol', 'Python')
('David', None)
('Emma', 'Machine Learning')


A LEFT JOIN returns all rows from the left table (students) and matching rows from the right table (enrollments).

If no match exists, SQL fills the columns with NULL.

| name  | course_name      |
| ----- | ---------------- |
| Alice | Python           |
| Alice | SQL              |
| Bob   | Data Science     |
| Carol | Python           |
| David | NULL             |
| Emma  | Machine Learning |


Aliases are short names for tables used to make queries easier to read.

Instead of writing:

students.student_id

we write:

s.student_id


In [5]:
#Query Without Aliases
cursor.execute("""SELECT students.name, enrollments.course_name
FROM students
INNER JOIN enrollments
ON students.student_id = enrollments.student_id;""")

Why Aliases Are Helpful

Aliases:

Make queries shorter

Improve readability

Avoid confusion when joining multiple tables

TASK2: Multiple table joins

In [7]:
cursor.execute('''SELECT s.name, e.course_name, g.score
FROM students AS s
LEFT JOIN enrollments AS e
ON s.student_id = e.student_id
LEFT JOIN grades AS g
ON s.student_id = g.student_id
AND e.course_name = g.course_name;''')

Explanation
Why LEFT JOIN is used instead of INNER JOIN

LEFT JOIN ensures all students are included, even if they:

have no enrollments

have no grades

If we used INNER JOIN, students without enrollments or grades would disappear from the result.

What happens to students with no enrollments?

Example: David

Result:

name	course_name	score
David	NULL	NULL

Because there is no enrollment record.

What happens to enrollments with no grades?

Example: Emma – Machine Learning

She enrolled but no grade exists.

Result:

name	course_name	score
Emma	Machine Learning	NULL

Why use both student_id AND course_name in join?
If we only join on student_id, SQL might match wrong course grades.

Using both ensures:

student + course = correct grade

TASK3: Why Normalize data

Normalization means splitting data into multiple related tables to avoid redundancy.
Problems:

Data duplication

More storage

Hard to update

Benefits of Normalization

Reduces data redundancy

*   Improves data consistency
*   Saves storage
*   Makes updates easier


That’s why we use:

students → student information

enrollments → course registrations

grades → scores



When to use each JOIN

Use INNER JOIN when:

You want only matching records from both tables.

Example: students who actually enrolled in courses.

Use LEFT JOIN when:

You want all records from the main table.

Even if related data does not exist.

Example: show all students even if they are not enrolled.

In [9]:
students_df = pd.DataFrame(students_data,columns=['student_id', 'name', 'email', 'city'])
enrollments_df = pd.DataFrame(enrollments_data,columns=['enrollment_id', 'student_id', 'course_name', 'enrollment_date'])
grades_df = pd.DataFrame(grades_data,columns=['grade_id', 'student_id', 'course_name', 'score'])

In [10]:
import pandas as pd

# First merge students with enrollments
result = pd.merge(
    students_df,
    enrollments_df,
    on="student_id",
    how="left"
)

# Then merge with grades
result = pd.merge(
    result,
    grades_df,
    on=["student_id", "course_name"],
    how="left"
)

print(result)

   student_id   name            email       city  enrollment_id  \
0           1  Alice  alice@email.com     Mumbai          101.0   
1           1  Alice  alice@email.com     Mumbai          102.0   
2           2    Bob    bob@email.com      Delhi          103.0   
3           3  Carol  carol@email.com  Bangalore          104.0   
4           4  David  david@email.com    Chennai            NaN   
5           5   Emma   emma@email.com       Pune          105.0   

        course_name enrollment_date  grade_id  score  
0            Python      2024-01-15     201.0   85.0  
1               SQL      2024-02-01     202.0   90.0  
2      Data Science      2024-01-20     203.0   78.0  
3            Python      2024-02-10       NaN    NaN  
4               NaN             NaN       NaN    NaN  
5  Machine Learning      2024-01-25       NaN    NaN  
